# CASIE (AAAI 2020) Token Classification - Test Set Evaluation
### การประเมินผลประสิทธิภาพโมเดลบน Test Set (100 ข่าวความมั่นคงปลอดภัยไซเบอร์)

สมุดโน้ตนี้สำหรับการทดสอบโมเดลและวัดผลบนเครื่องคอมพิวเตอร์ใน VS Code:
1. Single Model Evaluation: ทดสอบโมเดลเดี่ยวจาก 12 โมเดลที่ฝึกสอนเสร็จแล้ว
2. Dual Evaluation Standards:
   - Strict F1 (Exact Match): วัดความถูกต้องของ Span และ Label แบบตรงเป๊ะ 100%
   - Relaxed F1 (NIST TAC Overlap): เกณฑ์ Overlap ตามเปเปอร์ AAAI 2020 (Satyapanich et al.)
3. Detailed Metrics: Precision, Recall, Accuracy ทั้งภาพรวมและแยกราย Entity Type
4. Viterbi BIO Constrained Decoder: บังคับกฎการเปลี่ยนสถานะป้าย BIO ป้องกัน Invalid Transition
5. Multi-Model Ensemble: การรวมผลการทำนายด้วย Probability Averaging
6. Result Visualization: แสดงตาราง Ablation Study และกราฟเปรียบเทียบค่า F1 ทันทีจากผลที่คำนวณไว้แล้ว
7. Inference Playground: ทดสอบการทำนายข้อมูลด้วยประโยคใหม่แบบ Real-time

คำแนะนำเรื่องความเร็วในการรันบนเครื่อง (CPU Mode):
- หากต้องการดูตารางคะแนนรวมและกราฟวิจัยทันที: ข้ามไปรัน Step 6 ได้เลยทันที (โหลดจาก CSV ใน 0.1 วินาที)
- หากต้องการทดสอบรันโมเดลจริง: โค้ดตั้งค่า SAMPLE_LIMIT = 100 ประโยคเป็นค่าเริ่มต้น (รันเสร็จใน 5-10 วินาที)
- หากต้องการรันตรวจข้อสอบครบทั้ง 1,714 ประโยค: ปรับ SAMPLE_LIMIT = None (ใช้เวลาประมาณ 2-3 นาทีต่อโมเดลบน CPU)


## Step 1: Environment Setup and Path Configuration
ตรวจสอบอุปกรณ์ประมวลผล ตั้งค่า Multi-threading สำหรับ CPU และระบุตำแหน่งของโฟลเดอร์โมเดล


In [ ]:
import os
import sys
import json
import inspect
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from datasets import Dataset

# ตั้งค่าให้ PyTorch ใช้ CPU ครบทุก Core เพื่อเพิ่มความเร็วสูงสุด
cpu_cores = os.cpu_count() or 4
torch.set_num_threads(cpu_cores)

# ตรวจสอบ Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=" * 70)
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(f"Mode: CPU (Allocated {cpu_cores} threads for maximum performance)")
print("=" * 70)

# ค้นหาตำแหน่งโฟลเดอร์โมเดลในไดรฟ์ E:
MODEL_ROOT_CANDIDATES = [
    Path(r"E:\LPLLL22EE\NLP_MODELS_BACKUP"),
    Path(r"E:\LPLLL22EE\models"),
    Path(r"E:\LPLLL22EE"),
    Path(r"D:\Kasetsart\NLP\LPLLL\models"),
]

MODELS_DIR = None
for p in MODEL_ROOT_CANDIDATES:
    if p.exists() and (p / "master_ablation_results_v3.csv").exists():
        MODELS_DIR = p
        break
    elif p.exists() and len(list(p.glob("*_v3"))) > 0:
        MODELS_DIR = p
        break

if MODELS_DIR is None:
    raise FileNotFoundError("Directory not found: E:\\LPLLL22EE\\NLP_MODELS_BACKUP")

print(f"Models Directory: {MODELS_DIR}")

# ค้นหาตำแหน่งชุดข้อมูล
DATASET_CANDIDATES = [
    Path(r"D:\Kasetsart\NLP\LPLLL\dataset"),
    Path(r"E:\LPLLL22EE\dataset"),
    Path("dataset"),
]

DATASET_DIR = None
for p in DATASET_CANDIDATES:
    if p.exists() and (p / "test.json").exists():
        DATASET_DIR = p
        break

if DATASET_DIR is None:
    raise FileNotFoundError("Dataset not found: D:\\Kasetsart\\NLP\\LPLLL\\dataset")

print(f"Dataset Directory: {DATASET_DIR}")

# สแกนโมเดลที่มีในโฟลเดอร์
available_models = [d.name for d in MODELS_DIR.iterdir() if d.is_dir() and ((d / "config.json").exists() or (d / "experiment_config.json").exists())]
available_models = sorted(available_models)
print(f"\nFound {len(available_models)} trained models:")
for idx, m in enumerate(available_models, 1):
    print(f"  {idx:2d}. {m}")


## Step 2: Load Test Set and Label Mappings
โหลดชุดข้อมูลข้อสอบ test.json และ Mappings สำหรับป้าย BIO, POS และ General NER


In [ ]:
# โหลด Label Mappings
with open(DATASET_DIR / "label_mapping.json", "r", encoding="utf-8") as f:
    label_data = json.load(f)

if "label2id" in label_data:
    label2id = label_data["label2id"]
    id2label = {int(k): v for k, v in label_data.get("id2label", {}).items()}
    tag_list = label_data.get("tag_list", [id2label[i] for i in range(len(id2label))])
else:
    label2id = label_data
    id2label = {int(v): k for k, v in label2id.items()}
    tag_list = [id2label[i] for i in range(len(id2label))]

# โหลด Feature Mappings
pos2id = {}
ner_gen2id = {}
if (DATASET_DIR / "pos_mapping.json").exists():
    with open(DATASET_DIR / "pos_mapping.json", "r", encoding="utf-8") as f:
        pos_data = json.load(f)
    pos2id = pos_data.get("pos2id", pos_data)
if (DATASET_DIR / "ner_gen_mapping.json").exists():
    with open(DATASET_DIR / "ner_gen_mapping.json", "r", encoding="utf-8") as f:
        ner_gen_data = json.load(f)
    ner_gen2id = ner_gen_data.get("ner_gen2id", ner_gen_data)

# โหลด Test Set
with open(DATASET_DIR / "test.json", "r", encoding="utf-8") as f:
    test_raw = json.load(f)

total_tokens = sum(len(it["tokens"]) for it in test_raw)
print("=" * 70)
print(f"Loaded Test Set: {len(test_raw):,} sentences, {total_tokens:,} tokens")
print(f"Total BIO Labels: {len(label2id)}")
print(f"Label examples: {tag_list[:8]} ...")
print("=" * 70)


## Step 3: Model Architecture and Evaluation Logic
นิยามโครงสร้างสถาปัตยกรรมโมเดล อัลกอริทึม Viterbi BIO Decoding และฟังก์ชันวัดผล Strict / Relaxed F1


In [ ]:
class BertWithExtraFeaturesForTokenClassification(nn.Module):
    def __init__(self, model_name, num_labels, num_pos=0, num_ner_gen=0, pos_emb_dim=32, ner_emb_dim=32, dropout=0.1, o_weight=0.35):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.use_pos = (num_pos > 0)
        self.use_ner_gen = (num_ner_gen > 0)

        classifier_in_dim = hidden_size
        if self.use_pos:
            self.pos_embedding = nn.Embedding(num_pos, pos_emb_dim, padding_idx=0)
            classifier_in_dim += pos_emb_dim
        if self.use_ner_gen:
            self.ner_gen_embedding = nn.Embedding(num_ner_gen, ner_emb_dim, padding_idx=0)
            classifier_in_dim += ner_emb_dim

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(classifier_in_dim, num_labels)
        self.num_labels = num_labels
        self.o_weight = o_weight

        class_weights = torch.ones(num_labels, dtype=torch.float)
        class_weights[0] = float(o_weight)
        self.register_buffer("class_weights", class_weights)

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, pos_ids=None, ner_gen_ids=None, labels=None):
        bert_kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None and "token_type_ids" in inspect.signature(self.bert.forward).parameters:
            bert_kwargs["token_type_ids"] = token_type_ids

        bert_outputs = self.bert(**bert_kwargs)
        sequence_output = bert_outputs.last_hidden_state

        features = [sequence_output]
        if self.use_pos:
            if pos_ids is None: pos_ids = torch.zeros_like(input_ids)
            features.append(self.pos_embedding(pos_ids))
        if self.use_ner_gen:
            if ner_gen_ids is None: ner_gen_ids = torch.zeros_like(input_ids)
            features.append(self.ner_gen_embedding(ner_gen_ids))

        combined = torch.cat(features, dim=-1)
        combined = self.dropout(combined)
        combined = combined.to(self.classifier.weight.dtype)
        logits = self.classifier(combined)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device), ignore_index=-100)
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        return (loss, logits) if loss is not None else logits

class ExtraFeatureDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        first = features[0]
        batch = {}
        for k in ["input_ids", "attention_mask", "token_type_ids", "labels", "pos_ids", "ner_gen_ids"]:
            if k in first and first[k] is not None:
                pad_val = -100 if k == "labels" else (self.tokenizer.pad_token_id if k == "input_ids" else 0)
                max_len = max(len(f[k]) for f in features)
                batch[k] = torch.tensor([f[k] + [pad_val] * (max_len - len(f[k])) for f in features], dtype=torch.long)
        return batch

def tokenize_and_align_labels(examples, tokenizer, max_seq_length=128):
    tokenized_inputs = tokenizer(examples["tokens"], is_split_into_words=True, max_length=max_seq_length, truncation=True)
    labels = []
    pos_ids = []
    ner_gen_ids = []

    for i in range(len(examples["tokens"])):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        cur_pos = []
        cur_ner = []
        prev_word_idx = None

        for w_idx in word_ids:
            if w_idx is None:
                label_ids.append(-100)
                cur_pos.append(0)
                cur_ner.append(0)
            elif w_idx != prev_word_idx:
                label_ids.append(examples["ner_tags"][i][w_idx])
                cur_pos.append(examples["pos_tags"][i][w_idx] if "pos_tags" in examples else 0)
                cur_ner.append(examples["ner_gen_tags"][i][w_idx] if "ner_gen_tags" in examples else 0)
            else:
                label_ids.append(-100)
                cur_pos.append(examples["pos_tags"][i][w_idx] if "pos_tags" in examples else 0)
                cur_ner.append(examples["ner_gen_tags"][i][w_idx] if "ner_gen_tags" in examples else 0)
            prev_word_idx = w_idx

        labels.append(label_ids)
        pos_ids.append(cur_pos)
        ner_gen_ids.append(cur_ner)

    tokenized_inputs["labels"] = labels
    tokenized_inputs["pos_ids"] = pos_ids
    tokenized_inputs["ner_gen_ids"] = ner_gen_ids
    return tokenized_inputs

def extract_bio_spans(seq):
    spans = []
    current_type = None
    start = -1
    for i, tag in enumerate(seq):
        if tag.startswith("B-"):
            if current_type is not None: spans.append((current_type, start, i - 1))
            current_type = tag[2:]
            start = i
        elif tag.startswith("I-"):
            t = tag[2:]
            if current_type is not None and t == current_type: pass
            else:
                if current_type is not None: spans.append((current_type, start, i - 1))
                current_type = t
                start = i
        else:
            if current_type is not None:
                spans.append((current_type, start, i - 1))
                current_type = None
                start = -1
    if current_type is not None: spans.append((current_type, start, len(seq) - 1))
    return spans

def build_valid_transition_matrix(label2id):
    num_labels = len(label2id)
    id2label = {v: k for k, v in label2id.items()}
    valid_trans = np.zeros((num_labels, num_labels), dtype=bool)
    for u in range(num_labels):
        u_tag = id2label[u]
        for v in range(num_labels):
            v_tag = id2label[v]
            if v_tag == "O" or v_tag.startswith("B-"):
                valid_trans[u, v] = True
            elif v_tag.startswith("I-"):
                v_type = v_tag[2:]
                if (u_tag.startswith("B-") and u_tag[2:] == v_type) or (u_tag.startswith("I-") and u_tag[2:] == v_type):
                    valid_trans[u, v] = True
    return valid_trans

def viterbi_bio_decode(logits_seq, valid_trans, label2id):
    T, K = logits_seq.shape
    if valid_trans is None: return np.argmax(logits_seq, axis=-1)
    id2label = {v: k for k, v in label2id.items()}
    viterbi_dp = np.full((T, K), -1e9, dtype=np.float32)
    backpointers = np.zeros((T, K), dtype=np.int32)

    for k in range(K):
        tag = id2label[k]
        if tag == "O" or tag.startswith("B-"):
            viterbi_dp[0, k] = logits_seq[0, k]

    for t in range(1, T):
        for k in range(K):
            prev_scores = viterbi_dp[t - 1] + np.where(valid_trans[:, k], 0.0, -1e9)
            best_prev = np.argmax(prev_scores)
            viterbi_dp[t, k] = prev_scores[best_prev] + logits_seq[t, k]
            backpointers[t, k] = best_prev

    best_last = np.argmax(viterbi_dp[T - 1])
    path = [best_last]
    for t in range(T - 1, 0, -1):
        best_last = backpointers[t, best_last]
        path.append(best_last)
    path.reverse()
    return np.array(path)

def compute_span_metrics(true_predictions, true_labels):
    n_strict_tp = 0
    n_relaxed_tp = 0
    total_preds = 0
    total_labs = 0
    n_correct_tok = 0
    n_total_tok = 0

    per_class_stats = {}

    for p_seq, l_seq in zip(true_predictions, true_labels):
        p_spans = extract_bio_spans(p_seq)
        l_spans = extract_bio_spans(l_seq)
        total_preds += len(p_spans)
        total_labs += len(l_spans)

        # 1. Strict Set Match
        strict_matches = set(p_spans).intersection(set(l_spans))
        n_strict_tp += len(strict_matches)

        # 2. Relaxed Overlap Match (AAAI 2020 Benchmark)
        matched_l = set()
        for p_type, p_st, p_en in p_spans:
            if p_type not in per_class_stats:
                per_class_stats[p_type] = {"pred": 0, "true": 0, "tp_strict": 0, "tp_relaxed": 0}
            per_class_stats[p_type]["pred"] += 1

            for l_idx, (l_type, l_st, l_en) in enumerate(l_spans):
                if l_idx not in matched_l and p_type == l_type:
                    if max(p_st, l_st) <= min(p_en, l_en):
                        n_relaxed_tp += 1
                        matched_l.add(l_idx)
                        per_class_stats[p_type]["tp_relaxed"] += 1
                        break

        for l_type, l_st, l_en in l_spans:
            if l_type not in per_class_stats:
                per_class_stats[l_type] = {"pred": 0, "true": 0, "tp_strict": 0, "tp_relaxed": 0}
            per_class_stats[l_type]["true"] += 1

        for s_type, _, _ in strict_matches:
            if s_type in per_class_stats:
                per_class_stats[s_type]["tp_strict"] += 1

        for p_t, l_t in zip(p_seq, l_seq):
            if p_t == l_t: n_correct_tok += 1
            n_total_tok += 1

    s_p = n_strict_tp / total_preds if total_preds > 0 else 0.0
    s_r = n_strict_tp / total_labs if total_labs > 0 else 0.0
    s_f1 = (2 * s_p * s_r / (s_p + s_r)) if (s_p + s_r) > 0 else 0.0

    r_p = n_relaxed_tp / total_preds if total_preds > 0 else 0.0
    r_r = n_relaxed_tp / total_labs if total_labs > 0 else 0.0
    r_f1 = (2 * r_p * r_r / (r_p + r_r)) if (r_p + r_r) > 0 else 0.0
    acc = n_correct_tok / n_total_tok if n_total_tok > 0 else 0.0

    return {
        "strict_precision": round(s_p * 100, 2),
        "strict_recall": round(s_r * 100, 2),
        "strict_f1": round(s_f1 * 100, 2),
        "relaxed_precision": round(r_p * 100, 2),
        "relaxed_recall": round(r_r * 100, 2),
        "relaxed_f1": round(r_f1 * 100, 2),
        "accuracy": round(acc * 100, 2),
        "per_class": per_class_stats
    }

def load_custom_model(model_dir, cfg, device):
    model = BertWithExtraFeaturesForTokenClassification(
        model_name=cfg["model_name"],
        num_labels=cfg["num_labels"],
        num_pos=cfg.get("num_pos", 0),
        num_ner_gen=cfg.get("num_ner_gen", 0),
        o_weight=cfg.get("o_weight", 0.35)
    )
    weights_path = Path(model_dir) / "custom_model_weights.bin"
    if weights_path.exists() and weights_path.stat().st_size > 0:
        model.load_state_dict(torch.load(weights_path, map_location=device, weights_only=True))
    elif (Path(model_dir) / "model.safetensors").exists():
        from safetensors.torch import load_file
        state_dict = load_file(str(Path(model_dir) / "model.safetensors"))
        model.load_state_dict(state_dict, strict=False)
    elif (Path(model_dir) / "pytorch_model.bin").exists():
        model.load_state_dict(torch.load(Path(model_dir) / "pytorch_model.bin", map_location=device), strict=False)
    model.to(device)
    model.eval()
    return model

print("Model architecture and evaluation metrics loaded successfully.")


## Step 4: Single Model Evaluation
เลือกโมเดลที่ต้องการทดสอบ คำนวณ Strict F1, Relaxed F1 และแสดงตารางผลลัพธ์แยกตามประเภท Entity

หมายเหตุเรื่อง SAMPLE_LIMIT:
- ตั้งค่าเริ่มต้นไว้ที่ 100 ประโยค เพื่อให้รันเสร็จเร็วใน 5-10 วินาที
- หากต้องการตรวจครบทั้ง 1,714 ประโยค ให้เปลี่ยนเป็น SAMPLE_LIMIT = None


In [ ]:
# เลือกโมเดลที่ต้องการทดสอบ
SELECTED_MODEL = "roberta_base_pos_v3"

# จำนวนประโยคสำหรับทดสอบ (แนะนำ 100 เพื่อความรวดเร็วบน CPU, หรือใส่ None สำหรับชุดเต็ม 1,714 ประโยค)
SAMPLE_LIMIT = 100

model_dir = MODELS_DIR / SELECTED_MODEL
if not model_dir.exists():
    raise FileNotFoundError(f"Model directory not found: {model_dir}")

print(f"Loading model: {SELECTED_MODEL}...")
with open(model_dir / "experiment_config.json", "r", encoding="utf-8") as f:
    cfg = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(str(model_dir), use_fast=True)
model = load_custom_model(model_dir, cfg, device)

test_subset = test_raw[:SAMPLE_LIMIT] if SAMPLE_LIMIT else test_raw

test_hf = Dataset.from_dict({
    "tokens": [it["tokens"] for it in test_subset],
    "ner_tags": [it["ner_tags"] for it in test_subset],
    "pos_tags": [it["pos_tags"] for it in test_subset],
    "ner_gen_tags": [it["ner_gen_tags"] for it in test_subset]
})

# ใช้ max_seq_length = 128 เพื่อลดการคำนวณส่วนเกินบน CPU (ข้อสอบยาวเฉลี่ย 25 คำ)
test_tokenized = test_hf.map(
    lambda x: tokenize_and_align_labels(x, tokenizer, max_seq_length=128),
    batched=True,
    remove_columns=test_hf.column_names
)

data_collator = ExtraFeatureDataCollator(tokenizer)
training_args = TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=32, report_to="none")
trainer = Trainer(model=model, args=training_args, data_collator=data_collator)

print(f"Running inference on {len(test_subset)} sentences...")
preds_output = trainer.predict(test_tokenized)
logits = preds_output.predictions
labels = preds_output.label_ids

valid_trans = build_valid_transition_matrix(label2id)

true_predictions = []
true_labels = []

for sent_idx in range(len(logits)):
    sent_logits = logits[sent_idx]
    sent_labs = labels[sent_idx]
    valid_mask = (sent_labs != -100)
    if not np.any(valid_mask): continue

    valid_logits = sent_logits[valid_mask]
    valid_truth = sent_labs[valid_mask]

    pred_path = viterbi_bio_decode(valid_logits, valid_trans, label2id)
    true_predictions.append([tag_list[idx] for idx in pred_path])
    true_labels.append([tag_list[idx] for idx in valid_truth])

metrics = compute_span_metrics(true_predictions, true_labels)

print("\n" + "=" * 75)
print(f"Evaluation Results: {SELECTED_MODEL} (Sample Size: {len(test_subset)})")
print("=" * 75)
print(f"  Strict F1 (Exact Match):        {metrics['strict_f1']:.2f}%  (P: {metrics['strict_precision']:.2f}%, R: {metrics['strict_recall']:.2f}%)")
print(f"  Relaxed F1 (NIST TAC Benchmark): {metrics['relaxed_f1']:.2f}%  (P: {metrics['relaxed_precision']:.2f}%, R: {metrics['relaxed_recall']:.2f}%)")
print(f"  Token Accuracy:                 {metrics['accuracy']:.2f}%")
print("=" * 75)

# แสดงตารางผลลัพธ์แยกตาม Entity
class_rows = []
for c_name, c_stat in sorted(metrics["per_class"].items()):
    if c_stat["true"] > 0:
        c_p = (c_stat["tp_relaxed"] / c_stat["pred"] * 100) if c_stat["pred"] > 0 else 0.0
        c_r = (c_stat["tp_relaxed"] / c_stat["true"] * 100) if c_stat["true"] > 0 else 0.0
        c_f1 = (2 * c_p * c_r / (c_p + c_r)) if (c_p + c_r) > 0 else 0.0
        class_rows.append({
            "Cyber Entity Type": c_name,
            "Ground Truth": c_stat["true"],
            "Predicted": c_stat["pred"],
            "Relaxed Prec (%)": round(c_p, 1),
            "Relaxed Rec (%)": round(c_r, 1),
            "Relaxed F1 (%)": round(c_f1, 1),
        })

df_per_class = pd.DataFrame(class_rows).sort_values(by="Ground Truth", ascending=False)
display(df_per_class.head(15))


## Step 5: Multi-Model Ensemble Averaging
รวมผลการทำนายระดับคำ (Word-level Softmax Averaging) จากตัวแทน 3 สถาปัตยกรรม (RoBERTa, BERT, SecBERT) และถอดรหัสด้วย Viterbi Constrained Decoder


In [ ]:
ENSEMBLE_CANDIDATES = [
    "roberta_base_pos_v3",
    "bert_base_uncased_pos_v3",
    "SecBERT_pos_ner_v3"
]

# จำนวนตัวอย่างสำหรับการทดสอบเร็ว (ใส่ 100 เพื่อทดสอบระบบเร็ว หรือ None สำหรับประเมินครบ 1,714 ประโยค)
ENSEMBLE_SAMPLE_LIMIT = 100

ens_test_subset = test_raw[:ENSEMBLE_SAMPLE_LIMIT] if ENSEMBLE_SAMPLE_LIMIT else test_raw

print("=" * 75)
print(f"Multi-Model Ensemble on {len(ens_test_subset)} sentences: {', '.join(ENSEMBLE_CANDIDATES)}")
print("=" * 75)

all_model_word_probs = []
ground_truth_words = None

for m_name in ENSEMBLE_CANDIDATES:
    m_dir = MODELS_DIR / m_name
    print(f"Extracting probabilities from: {m_name}...")
    with open(m_dir / "experiment_config.json", "r", encoding="utf-8") as f:
        m_cfg = json.load(f)
    m_tok = AutoTokenizer.from_pretrained(str(m_dir), use_fast=True)
    m_model = load_custom_model(m_dir, m_cfg, device)

    t_hf = Dataset.from_dict({
        "tokens": [it["tokens"] for it in ens_test_subset],
        "ner_tags": [it["ner_tags"] for it in ens_test_subset],
        "pos_tags": [it["pos_tags"] for it in ens_test_subset],
        "ner_gen_tags": [it["ner_gen_tags"] for it in ens_test_subset]
    })
    t_tok = t_hf.map(
        lambda x: tokenize_and_align_labels(x, m_tok, max_seq_length=128),
        batched=True,
        remove_columns=t_hf.column_names
    )
    tr = Trainer(model=m_model, args=TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=32, report_to="none"), data_collator=ExtraFeatureDataCollator(m_tok))
    m_preds = tr.predict(t_tok)
    m_logits = m_preds.predictions
    m_labels = m_preds.label_ids

    sent_probs_list = []
    sent_truth_list = []
    for s_idx in range(len(m_logits)):
        s_log = m_logits[s_idx]
        s_lab = m_labels[s_idx]
        v_mask = (s_lab != -100)
        if not np.any(v_mask): continue
        v_probs = torch.softmax(torch.tensor(s_log[v_mask]), dim=-1).numpy()
        sent_probs_list.append(v_probs)
        if ground_truth_words is None:
            sent_truth_list.append(s_lab[v_mask])

    all_model_word_probs.append(sent_probs_list)
    if ground_truth_words is None:
        ground_truth_words = sent_truth_list

# Word-level Averaging
valid_trans = build_valid_transition_matrix(label2id)
ens_preds = []
ens_labels = []

for s_idx in range(len(ground_truth_words)):
    avg_probs = np.mean([model_probs[s_idx] for model_probs in all_model_word_probs], axis=0)
    avg_logits = np.log(avg_probs + 1e-12)
    path = viterbi_bio_decode(avg_logits, valid_trans, label2id)
    ens_preds.append([tag_list[idx] for idx in path])
    ens_labels.append([tag_list[idx] for idx in ground_truth_words[s_idx]])

ens_metrics = compute_span_metrics(ens_preds, ens_labels)

print("\n" + "=" * 75)
print(f"Ensemble Evaluation Results (Sample Size: {len(ens_test_subset)}):")
print("=" * 75)
print(f"  Strict F1 (Exact Match):        {ens_metrics['strict_f1']:.2f}%  (P: {ens_metrics['strict_precision']:.2f}%, R: {ens_metrics['strict_recall']:.2f}%)")
print(f"  Relaxed F1 (NIST TAC Benchmark): {ens_metrics['relaxed_f1']:.2f}%  (P: {ens_metrics['relaxed_precision']:.2f}%, R: {ens_metrics['relaxed_recall']:.2f}%)")
print(f"  Token Accuracy:                 {ens_metrics['accuracy']:.2f}%")
print("=" * 75)


## Step 6: Master Ablation Table and Performance Charts
แสดงตารางผลการทดลองเปรียบเทียบทุกโมเดลและพล็อตแผนภูมิแท่งเปรียบเทียบค่า F1 กับ Relaxed F1 (โหลดจากผลทางการที่คำนวณเสร็จสมบูรณ์แล้วใน 0.1 วินาที ไม่ต้องรอคำนวณใหม่)

### ความแตกต่างระหว่างเกณฑ์ Strict (F1 ทั่วไป) กับ Relaxed (NIST TAC Overlap Benchmark)
1. เกณฑ์ Strict (Precision, Recall, F1 มาตรฐานทั่วไป):
   - กฎการตรวจ: โมเดลต้องระบุประเภท Entity (Label) ได้ถูกต้อง และขอบเขตคำ (Start - End Token) ต้องตรงกับเฉลย 100% ทุกตำแหน่ง
   - ตัวอย่าง: สมมติเฉลย (Ground Truth) คือ "Russian hacker group Cozy Bear"
     - หากโมเดลทายว่า "Cozy Bear" (ตกคำว่า Russian hacker group) -> ระบบจะตรวจว่า ผิด (0 คะแนน)
     - โมเดลจะได้คะแนนต่อเมื่อทายครบถ้วนตรงเป๊ะ "Russian hacker group Cozy Bear" เท่านั้น
   - ลักษณะคะแนน: เป็นเกณฑ์ที่เข้มงวดที่สุดในงานวิจัย NLP Information Extraction คะแนน F1 ของโมเดลเดี่ยวจะอยู่ในช่วง 49% - 55% และ Ensemble ได้ 59.23%

2. เกณฑ์ Relaxed (Relaxed Precision, Relaxed Recall, Relaxed F1 ตามมาตรฐาน NIST TAC):
   - กฎการตรวจ: โมเดลต้องระบุประเภท Entity ถูกต้อง และขอบเขตคำมีส่วนเหลื่อมซ้อน (Overlap) กับเฉลยอย่างน้อย 1 Token
   - ตัวอย่าง: เฉลยคือ "Russian hacker group Cozy Bear"
     - หากโมเดลทายว่า "Cozy Bear" -> ระบบถือว่า ถูกต้อง (ได้คะแนน) เนื่องจากจับใจความและระบุตัวตนของผู้โจมตีได้ถูกต้อง
   - ลักษณะคะแนน: เป็นเกณฑ์มาตรฐานสากลที่เปเปอร์หลักของ รศ.ดร.ธนียา สัตยพาณิชย์ (CASIE, AAAI 2020) และ NIST TAC ใช้รายงานผลวิจัย คะแนน Relaxed F1 ของโมเดลเดี่ยวจะอยู่ในช่วง 64% - 68% และ Ensemble ได้ 72.13%


In [ ]:
csv_path = MODELS_DIR / "master_ablation_results_v3.csv"
if not csv_path.exists():
    csv_path = Path("master_ablation_results_v3.csv")

if csv_path.exists():
    df_results = pd.read_csv(csv_path)
    print("=" * 115)
    print("Official Master Ablation Table (Full Test Set 1,714 Sentences - 12 Models + Ensemble)")
    print("=" * 115)
    display(df_results)

    # พล็อตกราฟเปรียบเทียบ F1 (Strict) กับ Relaxed F1
    plt.figure(figsize=(15, 6), dpi=150)
    x = np.arange(len(df_results))
    width = 0.38

    plt.bar(x - width/2, df_results["F1 (%)"], width, label="F1 (Strict / Exact Match)", color="#2b5c8f")
    plt.bar(x + width/2, df_results["Relaxed F1 (%)"], width, label="Relaxed F1 (NIST TAC Overlap)", color="#c0392b")

    plt.xlabel("Experiment / Model Architecture", fontsize=11, fontweight="bold")
    plt.ylabel("F1 Score (%)", fontsize=11, fontweight="bold")
    plt.title("Ablation Study: F1 (Strict) vs Relaxed F1 on CASIE 100 Test News Articles", fontsize=12, fontweight="bold")
    plt.xticks(x, df_results["Experiment"], rotation=45, ha="right", fontsize=9)
    plt.ylim(0, 90)
    plt.legend(frameon=True, facecolor="white", loc="upper left")
    plt.grid(axis="y", linestyle="--", alpha=0.5)

    for i in range(len(df_results)):
        plt.text(i - width/2, df_results["F1 (%)"].iloc[i] + 1.2, f"{df_results['F1 (%)'].iloc[i]:.1f}", ha="center", fontsize=7.5, rotation=90)
        plt.text(i + width/2, df_results["Relaxed F1 (%)"].iloc[i] + 1.2, f"{df_results['Relaxed F1 (%)'].iloc[i]:.1f}", ha="center", fontsize=7.5, rotation=90)

    plt.tight_layout()
    chart_save_path = Path("ablation_f1_comparison_chart.png")
    plt.savefig(chart_save_path)
    print(f"\nSaved comparison chart to: {chart_save_path.resolve()}")
    plt.show()
else:
    print("File not found: master_ablation_results_v3.csv")


## Step 6.2: Plot All 12 Training and Validation Loss Curves
สร้างกราฟแสดงเส้นการลดลงของ Loss (Train Loss vs Validation Loss) ครบทั้ง 12 โมเดล
พร้อมแสดงจุด Early Stopping ที่ได้ค่า Validation Loss ต่ำที่สุด


In [ ]:
# 1. วาดกราฟ Train Loss vs Val Loss ครบทั้ง 12 โมเดล (ตาราง 3x4)
backbones = ["SecBERT", "bert_base_uncased", "roberta_base"]
features = ["none_v3", "pos_v3", "ner_v3", "pos_ner_v3"]
feature_titles = ["Baseline (None)", "+ POS", "+ NER", "+ POS & NER"]

fig, axes = plt.subplots(3, 4, figsize=(18, 11), dpi=150, sharey=True)

for r, bb in enumerate(backbones):
    for c, feat in enumerate(features):
        ax = axes[r, c]
        m_name = f"{bb}_{feat}"
        log_path = MODELS_DIR / m_name / "training_log.csv"
        
        if log_path.exists():
            df = pd.read_csv(log_path)
            epochs = df["epoch"].values
            t_loss = df["train_loss"].values
            v_loss = df["val_loss"].values
            
            best_idx = df["val_loss"].idxmin()
            best_epoch = epochs[best_idx]
            best_val = v_loss[best_idx]
            
            ax.plot(epochs, t_loss, label="Train Loss", color="#2b5c8f", linewidth=2, linestyle="-")
            ax.plot(epochs, v_loss, label="Val Loss", color="#c0392b", linewidth=2, linestyle="--")
            ax.scatter([best_epoch], [best_val], color="#e74c3c", s=80, zorder=5, label=f"Best: Ep {int(best_epoch)}")
            
            ax.set_title(f"{bb}\n{feature_titles[c]}", fontsize=10, fontweight="bold")
            ax.grid(True, linestyle=":", alpha=0.6)
            ax.set_xlabel("Epoch", fontsize=8)
            if c == 0:
                ax.set_ylabel("Loss", fontsize=9, fontweight="bold")
            if r == 0 and c == 0:
                ax.legend(loc="upper right", fontsize=8)
        else:
            ax.text(0.5, 0.5, f"{m_name}\nNot Found", ha="center", va="center")

plt.suptitle("Training and Validation Loss Curves Across 12 Models (CASIE AAAI 2020)", fontsize=13, fontweight="bold", y=0.99)
plt.tight_layout()
grid_path = Path("all_12_models_loss_curves.png")
plt.savefig(grid_path, bbox_inches="tight")
print(f"Saved 12-model loss curves grid to: {grid_path.resolve()}")
plt.show()

# 2. วาดกราฟเปรียบเทียบ Validation Loss ระหว่าง Feature แต่ละแบบในแต่ละ Backbone
bb_names = ["SecBERT (Domain Specific)", "BERT-base (Standard Baseline)", "RoBERTa-base (Meta SOTA)"]
feat_labels = ["Baseline (None)", "+ POS", "+ NER", "+ POS & NER"]
colors = ["#7f8c8d", "#2980b9", "#8e44ad", "#d35400"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), dpi=150, sharey=True)

for i, (bb, bb_label) in enumerate(zip(backbones, bb_names)):
    ax = axes[i]
    for feat, label, color in zip(features, feat_labels, colors):
        m_name = f"{bb}_{feat}"
        log_path = MODELS_DIR / m_name / "training_log.csv"
        if log_path.exists():
            df = pd.read_csv(log_path)
            ax.plot(df["epoch"], df["val_loss"], label=label, color=color, linewidth=2, marker="o", markersize=4)
    
    ax.set_title(bb_label, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch", fontsize=10)
    if i == 0:
        ax.set_ylabel("Validation Loss", fontsize=10, fontweight="bold")
    ax.grid(True, linestyle="--", alpha=0.5)
    ax.legend(frameon=True, facecolor="white", fontsize=9)

plt.suptitle("Validation Loss Convergence Across Feature Modes by Backbone", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
comp_path = Path("validation_loss_comparison_by_backbone.png")
plt.savefig(comp_path, bbox_inches="tight")
print(f"Saved backbone comparison to: {comp_path.resolve()}")
plt.show()


## Step 7: Live Prediction on Custom Text
ทดสอบการทำนายด้วยข้อความข่าวภัยคุกคามไซเบอร์ภาษาอังกฤษเพื่อสกัด Event Arguments (Attacker, Victim, Tool, Attack, Time)


In [ ]:
# กำหนดข้อความข่าวทดสอบ
SAMPLE_NEWS = "In August 2020, Russian hacker group Cozy Bear launched a spear-phishing attack against Microsoft using SolarWinds malware."

print(f"Input sentence:\n'{SAMPLE_NEWS}'\n")

predictor_model_name = "roberta_base_pos_v3"
p_dir = MODELS_DIR / predictor_model_name
with open(p_dir / "experiment_config.json", "r", encoding="utf-8") as f:
    p_cfg = json.load(f)

p_tok = AutoTokenizer.from_pretrained(str(p_dir), use_fast=True)
p_mod = load_custom_model(p_dir, p_cfg, device)

words = SAMPLE_NEWS.split()
encoding = p_tok(words, is_split_into_words=True, return_tensors="pt")
word_ids = encoding.word_ids(batch_index=0)

with torch.no_grad():
    input_ids = encoding["input_ids"].to(device)
    attn_mask = encoding["attention_mask"].to(device)
    out_logits = p_mod(input_ids=input_ids, attention_mask=attn_mask)
    if isinstance(out_logits, tuple): out_logits = out_logits[1]
    out_logits = out_logits[0].cpu().numpy()

subword_indices = []
prev_w = None
for idx, w_id in enumerate(word_ids):
    if w_id is not None and w_id != prev_w:
        subword_indices.append(idx)
        prev_w = w_id

first_subword_logits = out_logits[subword_indices]
valid_trans = build_valid_transition_matrix(label2id)
decoded_path = viterbi_bio_decode(first_subword_logits, valid_trans, label2id)
predicted_tags = [tag_list[idx] for idx in decoded_path]

spans = extract_bio_spans(predicted_tags)

print("=" * 65)
print("Extracted Cyber Event Arguments:")
print("=" * 65)
if len(spans) == 0:
    print("No event arguments detected.")
else:
    for entity_type, st, en in spans:
        entity_text = " ".join(words[st:en+1])
        print(f"  [{entity_type:<20}]  ->  \"{entity_text}\"")
print("=" * 65)
